## Build a Simple LLM Application with LCEL

In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

**After seeing this video, you'll have a high level overview of:**

• Using language models

• Using PromptTemplates and OutputParsers

• Using LangChain Expression Language (LCEL) to chain components together

• Debugging and tracing your application using LangSmith

• Deploying the application with LangServe

In [1]:
!pip install langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain_groq]


In [15]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
os.environ['LANGCHAIN_API_KEY']=os.getenv("LANGCHAIN_API_KEY")
os.environ['LANGCHAIN_TRACING_V2']="true"
os.environ['LANGCHAIN_PROJECT']=os.getenv("LANGCHAIN_PROJECT")

In [18]:
## Calling the groq model

model=ChatGroq(model="qwen/qwen3-32b")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x795932c93a10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x795932bec830>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [20]:
from langchain_core.messages import HumanMessage, SystemMessage

## Creating a list of messages to pass our model 
messages=[
    SystemMessage(content="Translate the following from English to Hindi"), #--> It help to pass a message to the system to behave in way
    HumanMessage(content="Hello, How are you?") # --> Passing the intruction to identify it as Human input
]

result=model.invoke(messages)

In [21]:
result

AIMessage(content='<think>\nOkay, the user wants to translate "Hello, How are you?" into Hindi. Let me start by recalling the basic greetings in Hindi. "Hello" is commonly translated as "नमस्ते" (Namaste), which is a standard greeting. Then "How are you?" can be translated as "आप कैसे हैं?" (Aap kaise ho?). I need to make sure the sentence structure is correct. In Hindi, the phrase order is similar to English here. Let me check if there\'s a more natural way to phrase it. Sometimes people might use "कैसे हो" (kaise ho) without "आप" (aap) if it\'s among friends, but since the original uses "how are you" which is formal, "आप" is appropriate. Also, the punctuation. The original has a comma, so in Hindi, it should be "नमस्ते, आप कैसे हैं?" with a comma. Let me verify if there\'s any regional variation. In some regions, people might use "तुम कैसे हो" (tum kaise ho) for informal contexts, but since the user hasn\'t specified, sticking with the formal version is safer. Also, confirming the sc

In [23]:
from langchain_core.output_parsers import StrOutputParser

parser=StrOutputParser()
parser.invoke(result)

'<think>\nOkay, the user wants to translate "Hello, How are you?" into Hindi. Let me start by recalling the basic greetings in Hindi. "Hello" is commonly translated as "नमस्ते" (Namaste), which is a standard greeting. Then "How are you?" can be translated as "आप कैसे हैं?" (Aap kaise ho?). I need to make sure the sentence structure is correct. In Hindi, the phrase order is similar to English here. Let me check if there\'s a more natural way to phrase it. Sometimes people might use "कैसे हो" (kaise ho) without "आप" (aap) if it\'s among friends, but since the original uses "how are you" which is formal, "आप" is appropriate. Also, the punctuation. The original has a comma, so in Hindi, it should be "नमस्ते, आप कैसे हैं?" with a comma. Let me verify if there\'s any regional variation. In some regions, people might use "तुम कैसे हो" (tum kaise ho) for informal contexts, but since the user hasn\'t specified, sticking with the formal version is safer. Also, confirming the script: "नमस्ते" is 

In [24]:
## Using LCEL and chaining the components

chain=model|parser

chain.invoke(messages)

'<think>\nOkay, the user wants me to translate "Hello, How are you?" into Hindi. Let me start by recalling the basic greetings in Hindi. "Hello" is commonly translated as "नमस्ते" (Namaste) or "हैलो" (Hello) which is used similarly to English. For "How are you?", the standard translation is "आप कैसे हैं?" (Aap kaise ho?). \n\nWait, should I use "आप" (Aap) or "तुम" (Tum)? Since "How are you?" is usually a polite greeting, "आप" is more appropriate for formal situations, whereas "तुम" is informal. The original sentence is in English and doesn\'t specify formality, but generally, when translating greetings, it\'s safer to use the formal version unless told otherwise.\n\nSo putting it together, "नमस्ते, आप कैसे हैं?" or "हैलो, आप कैसे हैं?" Both are correct. However, "नमस्ते" is more traditional and widely recognized in India, so maybe that\'s better. Let me double-check if there\'s any regional variation. In some regions, people might use different greetings, but "नमस्ते" is pretty univers

In [25]:
## Prompt Template

from langchain_core.prompts import ChatPromptTemplate

template="Translate the following into {language}"

prompt=ChatPromptTemplate(
    [
        ("system",template),("user","{text}")
    ]
)

In [27]:
prompt_template=prompt.invoke({"language":"French","text":"We are living in a AI era."})

In [28]:
prompt_template.to_messages()

[SystemMessage(content='Translate the following into French', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='We are living in a AI era.', additional_kwargs={}, response_metadata={})]

In [29]:
## Chaining together --> prompt, model & parser, using LCEL
chain_optimized = prompt|model|parser
template_result=chain_optimized.invoke({"language":"French","text":"Hello"})
print(f"Response we received in French for 'Hello':",template_result)

Response we received in French for 'Hello': <think>
Okay, the user wants to translate "Hello" into French. Let me think. The direct translation of "Hello" in French is "Bonjour". But wait, sometimes people might use "Salut" as well, which is more like a casual greeting. However, "Bonjour" is the standard and more formal way. Since the user just provided "Hello" without any context, it's safer to stick with "Bonjour". Let me double-check. Yes, "Bonjour" is the correct translation. No other nuances needed here. So the answer should be "Bonjour".
</think>

Bonjour
